In [7]:
# Libraries setup
import json
import math
import re
import time
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [13]:
# Run this code to setup the daily_sales data
ID = 42
rng = np.random.default_rng(seed=ID)

# base level per store + mild seasonality + noise
store_base = rng.integers(18000, 32000, size=8)
days = np.arange(42)
season = 1.0 + 0.05*np.sin(2*np.pi*days/14)

noise = rng.normal(0, 1200, size=(8, 42))
daily_sales = (store_base[:, None] * season[None, :] + noise).round(0)

# Inject a couple of store-specific anomalies deterministically
anomaly_store = ID % 8
anomaly_day = (ID * 3) % 42
daily_sales[anomaly_store, anomaly_day] *= 1.35  # spike
daily_sales[(anomaly_store+3)%8, (anomaly_day+11)%42] *= 0.55  # drop

daily_sales = daily_sales.astype(float)

promo_stores = np.array([1, 4, 6])
promo_days = (10, 24)  # inclusive
daily_sales.shape, promo_stores, promo_days

((8, 42), array([1, 4, 6]), (10, 24))

In [10]:
# First we compute total sales per store
total_sales_per_store = daily_sales.sum(axis=1)
total_sales_per_store

array([ 810576.  , 1207739.  , 1142728.35, 1010778.  , 1009993.  ,
       1246584.05,  806549.  , 1174893.  ])

array\(\[ 810576\., 1207739\., 1142728\.35, 1010778\., 1009993\., 1246584\.05, 806549\., 1174893\.\]\)

In [13]:
# then we find best store (index, total sales value)

best_store_idx = np.argmax(total_sales_per_store)
best_store_value = total_sales_per_store[best_store_idx]
best_store_idx, best_store_value

(5, 1246584.05)

We get Store 5, total sales = 1246584\.05€

In [16]:
# Grand total (all stores, all days)
grand_total_sales = total_sales_per_store.sum()

# Store share (%)
store_share_pct = 100 * total_sales_per_store / grand_total_sales
store_share_pct

array([ 9.6384231 , 14.36102164, 13.58799092, 12.01899147, 12.00965716,
       14.82292161,  9.59053872, 13.97045537])

We get an array of the percentages: \(\[ 9\.6384231 , 14\.36102164, 13\.58799092, 12\.01899147, 12\.00965716, 14\.82292161,  9\.59053872, 13\.97045537\]\)

In [37]:
num_stores, num_days = daily_sales.shape       # preparing containers to put Z scores and anomalies in
z_scores = np.zeros((num_stores, num_days))
anomalies = np.zeros((num_stores, num_days), dtype=bool)

for store in range(num_stores):        # for cycle to calculate mean & standard deviation for each store all days
    mean = daily_sales[store].mean()
    std = daily_sales[store].std()

    for day in range(num_days):        #  for cycle to calculate Z statistics for each store all days 
        z = (daily_sales[store, day] - mean) / std
        z_scores[store, day] = z

        if abs(z) > 2:                       # when Sales - Mean > 2 SD. we fall out of the 
            anomalies[store, day] = True     # region where 95% of data are distributed

for store in range(num_stores):
    days = np.where(anomalies[store])[0]
    if len(days) > 0:
        print(f"Store {store}: anomalous days {days}")

Store 0: anomalous days [ 0 18 31]
Store 2: anomalous days [0]
Store 3: anomalous days [12 32]
Store 4: anomalous days [ 4 17 18]
Store 5: anomalous days [11]
Store 6: anomalous days [17 18 25 37]
Store 7: anomalous days [ 2 32 37]


These are the days where each store sold too high or too low with respect to mean value: 

Store 0: anomalous days \[ 0 18 31\]
Store 2: anomalous days \[0\]
Store 3: anomalous days \[12 32\]
Store 4: anomalous days \[ 4 17 18\]
Store 5: anomalous days \[11\]
Store 6: anomalous days \[17 18 25 37\]
Store 7: anomalous days \[ 2 32 37\]

In [43]:
#This is easy! we just need to repeat the previous code, increasing z_score from 2 to 2.5
# I adjusted the variable anomalies to sec_anomalies, not to print the previous result!

sec_anomalies = np.zeros((num_stores, num_days), dtype=bool)

for store in range(num_stores):        # for cycle to calculate mean & standard deviation for each store all days
    mean = daily_sales[store].mean()
    std = daily_sales[store].std()

    for day in range(num_days):        #  for cycle to calculate Z statistics for each store all days 
        z = (daily_sales[store, day] - mean) / std
        z_scores[store, day] = z

        if abs(z) > 2.5:                  # when Sales - Mean > 2.5 SD. we fall out of the 
            sec_anomalies[store, day] = True  # region where 98.76% of data are distributed (extreme values)

for store in range(num_stores):
    days = np.where(sec_anomalies[store])[0]
    if len(days) > 0:
        print(f"Store {store}: anomalous days {days}")

Store 2: anomalous days [0]
Store 3: anomalous days [32]
Store 5: anomalous days [11]
Store 6: anomalous days [17 37]


Now we have less stores and days with extremely high or extremely low sales\.

Store 2: anomalous days \[0\]
Store 3: anomalous days \[32\]
Store 5: anomalous days \[11\]
Store 6: anomalous days \[17 37\]

In [55]:
# You can exectue this for anomalies or sec_anomalies
# np.argwhere finds anomalous positions

anomaly_positions = np.argwhere(anomalies) # for z_score> 2.5 use "sec_anomalies" instead of "anomalies"    

# create the list of tuples

anomaly_list = []

for store_idx, day_idx in anomaly_positions:
    value = daily_sales[store_idx, day_idx]
    z = z_scores[store_idx, day_idx]

    anomaly_list.append(
        (store_idx, day_idx, value, z)
    )
anomaly_list

[(0, 0, 16908.0, -2.305135956144752),
 (0, 18, 21654.0, 2.2696087711575728),
 (0, 31, 21542.0, 2.16165019352212),
 (2, 0, 37450.350000000006, 4.745867481765318),
 (3, 12, 20642.0, -2.4140924853399177),
 (3, 32, 27875.0, 2.685324120494507),
 (4, 4, 27130.0, 2.1459756778226753),
 (4, 17, 27216.0, 2.2058462560263528),
 (4, 18, 27303.0, 2.2664130037440264),
 (5, 11, 15857.050000000001, -4.913720720437835),
 (6, 17, 22651.0, 2.7855770265768833),
 (6, 18, 22027.0, 2.2813786004614847),
 (6, 25, 16173.0, -2.448713685563427),
 (6, 37, 15372.0, -3.0959299344711746),
 (7, 2, 30921.0, 2.1663547901118134),
 (7, 32, 30930.0, 2.1729699345161007),
 (7, 37, 25218.0, -2.0254417140712926)]

This list contains the z\-score for values 2 standard deviation away from mean

If you want the z\-statistics for z \> 2\.5 sd, use sec\_anomalies instead of anomalies in the code

\[\(0, 0, 16908\.0, \-2\.305135956144752\),
 \(0, 18, 21654\.0, 2\.2696087711575728\),
 \(0, 31, 21542\.0, 2\.16165019352212\),
 \(2, 0, 37450\.350000000006, 4\.745867481765318\),
 \(3, 12, 20642\.0, \-2\.4140924853399177\),
 \(3, 32, 27875\.0, 2\.685324120494507\),
 \(4, 4, 27130\.0, 2\.1459756778226753\),
 \(4, 17, 27216\.0, 2\.2058462560263528\),
 \(4, 18, 27303\.0, 2\.2664130037440264\),
 \(5, 11, 15857\.050000000001, \-4\.913720720437835\),
 \(6, 17, 22651\.0, 2\.7855770265768833\),
 \(6, 18, 22027\.0, 2\.2813786004614847\),
 \(6, 25, 16173\.0, \-2\.448713685563427\),
 \(6, 37, 15372\.0, \-3\.0959299344711746\),
 \(7, 2, 30921\.0, 2\.1663547901118134\),
 \(7, 32, 30930\.0, 2\.1729699345161007\),
 \(7, 37, 25218\.0, \-2\.0254417140712926\)\]

In [16]:
promo_stores = np.array([1, 4, 6])
promo_days = (10, 24)  # inclusive
start_day, end_day = promo_days

incremental_sales = 0.0
for store in promo_stores:
    for day in range(start_day, end_day + 1):
        incremental_sales += 0.03 * daily_sales[store, day]

incremental_sales


32608.620000000003

The result is 32,608\.62€

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=15786b62-0388-4a37-a980-cd24fdd44c55' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>